# ImageCAS Large Dataset — EDA Notebook
**Project:** Artery Segmentation and Stenosis Detection Using Advanced Deep Learning Techniques

This notebook performs EDA on your actual ImageCAS files once the dataset is available in the configured folder. It inventories files, matches images and masks, checks NIfTI metadata and mask labels, generates summaries, and visualizes sample overlays.

**Dataset access:** ImageCAS contains approximately 1,000 3D coronary CTA scans. The official repository points to Kaggle and notes that the authors can provide the latest download link/password: https://github.com/XiaoweiXu/ImageCAS-A-Large-Scale-Dataset-and-Benchmark-for-Coronary-Artery-Segmentation-based-on-CT  
Kaggle: https://www.kaggle.com/datasets/xiaoweixumedicalai/imagecas

I cannot authenticate to your Kaggle account or pull the entire large dataset into this chat environment. Attach ImageCAS as a Kaggle Notebook input or download/extract it locally, then run this notebook. No statistics are fabricated here.

**Important:** ImageCAS is primarily an artery-segmentation dataset. Segmentation masks alone are not stenosis-specific annotations.

## 1. Install and import packages
If needed, uncomment the install command. This notebook reads NIfTI (`.nii` / `.nii.gz`) files.

In [ ]:
# If needed:
# %pip install nibabel pandas numpy matplotlib tqdm

from pathlib import Path
import re, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import nibabel as nib

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (8, 5)
print("Ready. nibabel version:", nib.__version__)

## 2. Set dataset path
Set `DATA_ROOT` to the extracted ImageCAS folder. Example paths:
- Kaggle: `/kaggle/input/imagecas`
- Windows: `D:/datasets/ImageCAS`
The notebook searches recursively because folder layouts can vary.

In [ ]:
DATA_ROOT = Path("/kaggle/input/imagecas")  # EDIT THIS for your environment
# Windows example: DATA_ROOT = Path(r"D:\datasets\ImageCAS")

if not DATA_ROOT.exists():
    print(f"Dataset path not found: {DATA_ROOT}. Update DATA_ROOT first.")
else:
    print("Dataset root:", DATA_ROOT.resolve())
    print("Top-level entries:", [p.name for p in list(DATA_ROOT.iterdir())[:20]])

## 3. File inventory and candidate classification

In [ ]:
EXTS = (".nii.gz", ".nii", ".mha", ".mhd", ".nrrd")
def supported(p):
    return any(p.name.lower().endswith(e) for e in EXTS)
def classify(p):
    n = p.name.lower()
    if any(x in n for x in ("label", "mask", "seg", "annotation", "gt")):
        return "label_candidate"
    if any(x in n for x in ("img", "image", "ct", "scan", "volume")):
        return "image_candidate"
    return "unknown_candidate"

all_files = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and supported(p)) if DATA_ROOT.exists() else []
print("Supported medical-image files:", len(all_files))
inventory = pd.DataFrame([{
    "relative_path": str(p.relative_to(DATA_ROOT)), "filename": p.name,
    "parent_folder": p.parent.name, "size_MB": round(p.stat().st_size / 1024**2, 3),
    "candidate_type": classify(p),
    "format": ".nii.gz" if p.name.lower().endswith(".nii.gz") else p.suffix.lower()
} for p in all_files])
display(inventory.head(30))
inventory.to_csv("imagecas_file_inventory.csv", index=False)
print("Saved imagecas_file_inventory.csv")

## 4. Pair images and labels
The common ImageCAS layout uses `img.nii.gz` and `label.nii.gz` in a case folder. Pairing below uses the same-folder rule and flags folders needing manual review. Always confirm the pairs using shape checks and visual overlays.

In [ ]:
nifti_files = [p for p in all_files if p.name.lower().endswith((".nii", ".nii.gz"))]
pairs_list = []
for parent in sorted(set(p.parent for p in nifti_files)):
    files = [p for p in nifti_files if p.parent == parent]
    imgs = [p for p in files if classify(p) == "image_candidate"]
    labs = [p for p in files if classify(p) == "label_candidate"]
    if imgs or labs:
        pairs_list.append({
            "case_folder": str(parent.relative_to(DATA_ROOT)),
            "image_path": str(imgs[0].relative_to(DATA_ROOT)) if len(imgs) == 1 else "",
            "label_path": str(labs[0].relative_to(DATA_ROOT)) if len(labs) == 1 else "",
            "image_candidates": len(imgs), "label_candidates": len(labs),
            "pair_status": "paired_by_folder" if len(imgs) == 1 and len(labs) == 1 else "review_needed"
        })
pairs = pd.DataFrame(pairs_list)
display(pairs.head(30))
pairs.to_csv("imagecas_pairing_report.csv", index=False)
if not pairs.empty:
    display(pairs["pair_status"].value_counts().to_frame("count"))
print("Saved imagecas_pairing_report.csv")

## 5. NIfTI header metadata (without loading all image voxels)

In [ ]:
meta_rows = []
for p in tqdm(nifti_files, desc="Reading headers"):
    try:
        nii = nib.load(str(p), mmap=True)
        meta_rows.append({
            "relative_path": str(p.relative_to(DATA_ROOT)),
            "shape": str(tuple(int(x) for x in nii.shape)),
            "ndim": len(nii.shape),
            "spacing_mm": str(tuple(round(float(x), 5) for x in nii.header.get_zooms())),
            "dtype": str(nii.header.get_data_dtype()),
            "orientation": "".join(nib.aff2axcodes(nii.affine)),
            "voxel_count": int(np.prod(nii.shape)),
            "estimated_float32_MB": round(np.prod(nii.shape) * 4 / 1024**2, 2),
            "error": ""
        })
    except Exception as e:
        meta_rows.append({"relative_path": str(p.relative_to(DATA_ROOT)), "error": repr(e)})
metadata = pd.DataFrame(meta_rows)
display(metadata.head(30))
metadata.to_csv("imagecas_nifti_metadata.csv", index=False)
if not metadata.empty:
    print("Shape distribution"); display(metadata["shape"].value_counts(dropna=False).head(20).to_frame("file_count"))
    print("Orientation distribution"); display(metadata["orientation"].value_counts(dropna=False).to_frame("file_count"))
print("Saved imagecas_nifti_metadata.csv")

## 6. Image-mask compatibility and annotation checks
Loads one mask at a time to limit memory use. This can take time across the full dataset.

In [ ]:
compat_rows = []
valid_pairs = pairs[pairs["pair_status"] == "paired_by_folder"] if not pairs.empty else pd.DataFrame()
for _, row in tqdm(valid_pairs.iterrows(), total=len(valid_pairs), desc="Checking pairs"):
    ip, lp = DATA_ROOT / row["image_path"], DATA_ROOT / row["label_path"]
    try:
        im, lm = nib.load(str(ip), mmap=True), nib.load(str(lp), mmap=True)
        lab = np.asanyarray(lm.dataobj)
        vals = np.unique(lab)
        same_shape = tuple(im.shape) == tuple(lm.shape)
        same_spacing = np.allclose(im.header.get_zooms(), lm.header.get_zooms(), atol=1e-4, rtol=0)
        same_affine = np.allclose(im.affine, lm.affine, atol=1e-3, rtol=0)
        fg = int(np.count_nonzero(lab))
        compat_rows.append({
            "case_folder": row["case_folder"], "image_shape": str(tuple(im.shape)),
            "label_shape": str(tuple(lm.shape)), "same_shape": same_shape,
            "same_spacing": same_spacing, "same_affine": same_affine,
            "unique_label_values": str(vals[:100].tolist()), "unique_label_count": len(vals),
            "foreground_voxels": fg, "empty_mask": fg == 0,
            "status": "ok" if same_shape and same_spacing and same_affine and fg > 0 else "review"
        })
        del lab
    except Exception as e:
        compat_rows.append({"case_folder": row["case_folder"], "status": "read_error", "error": repr(e)})
compatibility = pd.DataFrame(compat_rows)
display(compatibility.head(30))
compatibility.to_csv("imagecas_pair_compatibility.csv", index=False)
if not compatibility.empty and "status" in compatibility:
    display(compatibility["status"].value_counts().to_frame("count"))
print("Saved imagecas_pair_compatibility.csv")

## 7. Visualize original scan, mask and overlay
Chooses the slice with the most foreground label voxels along the selected axis. Inspect anatomy/orientation manually; this is a visualization aid, not clinical interpretation.

In [ ]:
def show_overlay(pair_index=0, axis=2, slice_index=None):
    vp = pairs[pairs["pair_status"] == "paired_by_folder"].reset_index(drop=True)
    if len(vp) == 0:
        raise ValueError("No automatically paired image-mask cases. Review the pairing report.")
    row = vp.iloc[pair_index]
    im_nii = nib.load(str(DATA_ROOT / row["image_path"]), mmap=True)
    lb_nii = nib.load(str(DATA_ROOT / row["label_path"]), mmap=True)
    image, label = np.asarray(im_nii.dataobj), np.asarray(lb_nii.dataobj)
    if image.shape != label.shape:
        raise ValueError(f"Shape mismatch: {image.shape} vs {label.shape}")
    if slice_index is None:
        counts = np.count_nonzero(label, axis=tuple(i for i in range(label.ndim) if i != axis))
        slice_index = int(np.argmax(counts))
    if axis == 0: im2, lb2 = image[slice_index,:,:], label[slice_index,:,:]
    elif axis == 1: im2, lb2 = image[:,slice_index,:], label[:,slice_index,:]
    elif axis == 2: im2, lb2 = image[:,:,slice_index], label[:,:,slice_index]
    else: raise ValueError("axis must be 0, 1, or 2")
    finite = im2[np.isfinite(im2)]
    vmin, vmax = (np.percentile(finite, 1), np.percentile(finite, 99)) if finite.size else (None, None)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(np.rot90(im2), cmap="gray", vmin=vmin, vmax=vmax); ax[0].set_title("Original CT")
    ax[1].imshow(np.rot90(lb2), cmap="nipy_spectral", interpolation="nearest"); ax[1].set_title("Annotation mask")
    ax[2].imshow(np.rot90(im2), cmap="gray", vmin=vmin, vmax=vmax)
    ax[2].imshow(np.rot90(np.ma.masked_where(lb2 == 0, lb2)), cmap="autumn", alpha=.55, interpolation="nearest")
    ax[2].set_title("Overlay")
    for a in ax: a.axis("off")
    fig.suptitle(f"Case {row['case_folder']} | axis={axis}, slice={slice_index}")
    plt.tight_layout(); plt.savefig("eda_sample_overlay.png", dpi=160, bbox_inches="tight"); plt.show()

if not valid_pairs.empty:
    show_overlay(pair_index=0, axis=2)
else:
    print("No valid pair found for overlay.")

## 8. Summary charts

In [ ]:
if not inventory.empty:
    inventory["candidate_type"].value_counts().plot(kind="bar", title="Files by candidate type")
    plt.xlabel("Candidate type"); plt.ylabel("File count"); plt.xticks(rotation=25, ha="right")
    plt.tight_layout(); plt.savefig("eda_file_types.png", dpi=160); plt.show()
if not metadata.empty and "voxel_count" in metadata:
    v = pd.to_numeric(metadata["voxel_count"], errors="coerce").dropna()
    if len(v):
        plt.figure(); plt.hist(v / 1e6, bins=30)
        plt.title("NIfTI volume-size distribution"); plt.xlabel("Voxel count (millions)"); plt.ylabel("File count")
        plt.tight_layout(); plt.savefig("eda_voxel_count_distribution.png", dpi=160); plt.show()
if not compatibility.empty and "foreground_voxels" in compatibility:
    v = pd.to_numeric(compatibility["foreground_voxels"], errors="coerce").dropna()
    if len(v):
        plt.figure(); plt.hist(v / 1e6, bins=30)
        plt.title("Mask foreground voxel distribution"); plt.xlabel("Non-zero voxels (millions)"); plt.ylabel("Case count")
        plt.tight_layout(); plt.savefig("eda_mask_foreground_distribution.png", dpi=160); plt.show()

## 9. Export a run summary

In [ ]:
summary = {
    "dataset_root": str(DATA_ROOT),
    "medical_image_files_found": int(len(all_files)),
    "nifti_files_found": int(len(nifti_files)),
    "candidate_image_files": int(sum(classify(p) == "image_candidate" for p in nifti_files)),
    "candidate_label_files": int(sum(classify(p) == "label_candidate" for p in nifti_files)),
    "case_folders_with_candidates": int(len(pairs)),
    "automatically_paired_cases": int((pairs["pair_status"] == "paired_by_folder").sum()) if not pairs.empty else 0,
    "compatibility_rows": int(len(compatibility)),
    "note": "These are run-time findings from files available under DATA_ROOT; verify all pairs and annotation meanings."
}
with open("imagecas_eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
print("Saved imagecas_eda_summary.json")

## 10. How to write conclusions after running

Report only values generated by your run:
1. Count of scans, annotation files, and verified image-mask pairs.
2. Distribution of dimensions, voxel spacing, data types and orientations.
3. Number of empty masks, unmatched pairs, and shape/spacing/affine mismatches.
4. Whether overlays visually align with coronary arteries.
5. Any preprocessing needed (for example, orientation normalization or resampling), justified by results.
6. Whether the dataset includes stenosis-specific labels. If it only has artery masks, it supports segmentation but cannot by itself supervise lesion-level stenosis detection.

**Limitations:** This notebook is for research data inspection, not clinical diagnosis. Split train/validation/test by patient/case to prevent leakage. Keep patient identifiers private.

**Dataset citation:** Zeng et al., “ImageCAS: A large-scale dataset and benchmark for coronary artery segmentation based on computed tomography angiography images,” *Computerized Medical Imaging and Graphics*, 2023, 109:102287. Confirm bibliographic details against the official publication before submission.